In [6]:

from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def getPdfDocs(pdfDir):
    allDocs = []
    pobj = Path(pdfDir)
    if not pobj.is_dir():
        print(f"Dir Not Found")
        return None
    print(pobj.rglob("**/*.pdf"))
    pdf_files = pobj.rglob("**/*.pdf")
    # procces the pdf files 
    for pdf_file in pdf_files:
        print(f"Processing :{pdf_file}")
        loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()
        print(f"Loaded {len(docs)} pages")
        for doc in docs:
            doc.metadata["file_name"] = pdf_file.stem
            doc.metadata["file_type"] = pdf_file.suffix
        allDocs.extend(docs)
    return allDocs
all_docs = getPdfDocs("../doc_files/")
all_docs

<generator object Path.rglob at 0x000002573E1018B0>
Processing :..\doc_files\SowmyaG_Resume.pdf
Loaded 1 pages
Processing :..\doc_files\SPL.pdf
Loaded 6 pages
Processing :..\doc_files\VivekanandNaikCV.pdf
Loaded 1 pages
Processing :..\doc_files\Vivekanand_Naik_Resume.pdf
Loaded 1 pages


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-08-03T16:14:20+05:30', 'author': 'Un-named', 'moddate': '2026-08-03T16:14:20+05:30', 'source': '..\\doc_files\\SowmyaG_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': 'SowmyaG_Resume', 'file_type': '.pdf'}, page_content='SOWMYA G \nSoftware Developer | Python & Django Developer | Backend Development \n+91-8147588578 | g34402284@gmail.com | github.com/SowmyaGopal12 | linkedin//Sowmya G \nSUMMARY \nComputer Science undergraduate skilled in Python, Java, and Django, with hands -on experience building full-stack style applications \nusing object-oriented design and clean UI principles. Experienced building CRUD -driven applications with structured data \nmanagement, exception handling, and input validation. Strong foundation in Object -Oriented Programming, file handling, and version \ncontrol through Git and GitHub. \nTECHNICAL SKILLS \nLanguages: Pyth

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


# a text splitter function
def split_doc(docs, chunkSize=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunkSize,
        chunk_overlap = chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
        length_function =len
    )
    splitted_doc = text_splitter.split_documents(docs)
    print(f"splitted {len(docs)} Docs into {len(splitted_doc)}")
    # print(f"Content: {splitted_doc[0].page_content[:200]}")
    return splitted_doc
splitted_docs = split_doc(all_docs)
splitted_docs

splitted 9 Docs into 23


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-08-03T16:14:20+05:30', 'author': 'Un-named', 'moddate': '2026-08-03T16:14:20+05:30', 'source': '..\\doc_files\\SowmyaG_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': 'SowmyaG_Resume', 'file_type': '.pdf'}, page_content='SOWMYA G \nSoftware Developer | Python & Django Developer | Backend Development \n+91-8147588578 | g34402284@gmail.com | github.com/SowmyaGopal12 | linkedin//Sowmya G \nSUMMARY \nComputer Science undergraduate skilled in Python, Java, and Django, with hands -on experience building full-stack style applications \nusing object-oriented design and clean UI principles. Experienced building CRUD -driven applications with structured data \nmanagement, exception handling, and input validation. Strong foundation in Object -Oriented Programming, file handling, and version \ncontrol through Git and GitHub. \nTECHNICAL SKILLS \nLanguages: Pyth

In [8]:
from sentence_transformers import SentenceTransformer
from typing_extensions import List, Any

class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = None
        self.model_name = model_name
        self._load_model()
    
    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Dimension : {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model: {self.model_name} : {e}")
            raise ValueError("Model Cannot Be Loaded...")
        
    def generate_embedding(self, texts: List[str]):
        try:
            encodings = self.model.encode(texts, show_progress_bar=True)
            return encodings
        except Exception as e :
            print(e)
            
    def get_embedding_dimension(self):
        if not self.model:
            raise ValueError("Model Doesn't Exist")
        return self.model.get_embedding_dimension()

embedding_manager = EmbeddingManager()
embedding_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4269.75it/s]


Model Dimension : 384


In [9]:
import chromadb
import os
import uuid
from typing_extensions import List, Any
import numpy as np

# Vector Store
class VectorStore:
    def __init__(self, collection_name: str = "PDF_Collection", persist_dir: str = "./vector_store"):
        self.collection_name = collection_name
        self.persist_dir = persist_dir
        self.collection = None
        self.client = None
        self._load_vectorDB()
    
    # load the colleciton and client
    def _load_vectorDB(self):
        # initialize collection and client
        os.makedirs(self.persist_dir, exist_ok=True)
        try:
            self.client = chromadb.PersistentClient(path=self.persist_dir)
            # self.client.delete_collection(name=self.collection_name)
            self.collection = self.client.get_or_create_collection(
                self.collection_name, 
                configuration={
                  "hnsw": {
                      "space": "cosine"
                  }
                },
                metadata={
                    "description": "PDF files for RAG"
                    }
                )
            print(f"Vector Store initialized successfully {self.collection_name}")
            print(f"Existing Documents in collection {self.collection.count()}")
        except Exception as e:
            print(e)    
    # Add docs to vector store 
    def add_docs(self, docs: List[Any], embeddings: np.ndarray):
        # prepare data 
        doc_ids = []
        doc_contents = []
        doc_embeddings = []
        metadatas = []
        for i, (doc, embedding) in enumerate(zip(docs, embeddings)):
            # unique id for each doc
            uid = f"{uuid.uuid4().hex[:8]}_{i}"
            doc_ids.append(uid)
            
            # prepare page content
            doc_contents.append(doc.page_content)
            
            # embeddings 
            doc_embeddings.append(embedding.tolist())
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["document_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            print(f"metadata: {metadata}")
            
        # add fields to vector Store 
        try:
            self.collection.add(
                ids=doc_ids,
                documents=doc_contents,
                embeddings=doc_embeddings,
                metadatas=metadatas
            )
            print(f"Successfully added {len(docs)} into vector store...")
        except Exception as e :
            print(e)


vectorstore_manager = VectorStore()

Vector Store initialized successfully PDF_Collection
Existing Documents in collection 0


In [10]:
texts = [doc.page_content for doc in splitted_docs]
chunks_embeddings = embedding_manager.generate_embedding(texts)

vectorstore_manager.add_docs(splitted_docs, chunks_embeddings)

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.25it/s]

metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-08-03T16:14:20+05:30', 'author': 'Un-named', 'moddate': '2026-08-03T16:14:20+05:30', 'source': '..\\doc_files\\SowmyaG_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': 'SowmyaG_Resume', 'file_type': '.pdf', 'document_index': 0, 'content_length': 966}
metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-08-03T16:14:20+05:30', 'author': 'Un-named', 'moddate': '2026-08-03T16:14:20+05:30', 'source': '..\\doc_files\\SowmyaG_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_name': 'SowmyaG_Resume', 'file_type': '.pdf', 'document_index': 1, 'content_length': 970}
metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-08-03T16:14:20+05:30', 'author': 'Un-named', 'moddate': '2026-08-03T16:14:20+05:30', 'source': '..\\doc_files\\SowmyaG_Resume.pdf', 'tot

In [11]:
from typing_extensions import Dict
# Reterival Pipeline
class ReterivalManager:
    def __init__(self, embedding_manager: EmbeddingManager, vector_store: VectorStore):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store
    
    def reteriveContext(self, query: str, top_k: int, threshold: float = 0.0) -> list[Dict[str, Any]]:
        # Generate the qurey embedding 
        query_embedding = self.embedding_manager.generate_embedding([query])
        
        # serach in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=query_embedding.tolist(),
                n_results=top_k
            )
            reterived_docs = []
            if results["documents"] and results["documents"][0]:
                docs = results["documents"][0]
                ids = results["ids"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                
                for i, (id_, doc, metadata, distance) in enumerate(zip(ids, docs, metadatas, distances)):
                    # convert distanbce to simlarity score (Chromadb uses cosine distance)
                    similarity_score = 1 - distance
                    if similarity_score >= threshold:
                        reterived_docs.append({
                            "id": id_,
                            "content": doc,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                print(f"Reterived {len(reterived_docs)}")
            else:
                print("no document Reterived ")
            return reterived_docs
        except Exception as e:
            print(e)

reterival = ReterivalManager(embedding_manager, vectorstore_manager)
reterival

In [12]:
reterival.reteriveContext("hello", 10)

Batches: 100%|██████████| 1/1 [00:00<00:00, 104.75it/s]

Reterived 10


[{'id': '929948bd_11',
  'content': 'Photographs:',
  'metadata': {'content_length': 12,
   'title': 'Mock_Press_Report',
   'page': 5,
   'page_label': '6',
   'file_type': '.pdf',
   'creationdate': '2026-02-20T19:24:23+05:30',
   'file_name': 'SPL',
   'total_pages': 6,
   'document_index': 11,
   'moddate': '2026-02-20T19:24:23+05:30',
   'source': '..\\doc_files\\SPL.pdf',
   'producer': 'Microsoft® Word 2021',
   'author': 'Vivekanand Naik',
   'creator': 'Microsoft® Word 2021'},
  'similarity_score': 0.23551911115646362,
  'distance': 0.7644808888435364,
  'rank': 1},
 {'id': '45c8c543_22',
  'content': '● Organized a college-level hackathon (registrations, evaluation, logistics, judging) and anchored technical/cultural events, \ndemonstrating leadership and public speaking. \nADDITIONAL INFORMATION \nAreas of Interest: Agentic AI, AI/ML, Full Stack Development, Data Science    |    Interests: Travelling, Teaching, Cooking',
  'metadata': {'moddate': '2026-09-11T15:29:45+05:30',

In [16]:
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from typing import Any, List, Tuple, Dict
import re
load_dotenv()

class AdvanceRagImplementation:
    # init
    def __init__(self, agent: Any, reteriver: ReterivalManager):
        self.agent = agent
        self.reteriver = reteriver
        self.history = []
        
    # reterival
    def _reterive_doc(self, query: str, top_k: int, threshold: float)->list[Dict[str, Any]]:
        return self.reteriver.reteriveContext(query=query, top_k=top_k, threshold=threshold)
    
    # context with citations and source 
    def _build_context(self, reterived_doc: Any) -> Tuple[str, List[Dict]]:
        context = "".join(
            f"[{i}]\n{doc['content']}"
            for i, doc in enumerate(reterived_doc, start=1))
        
        sources = [
            {
                "citation_id": i,
                "chunk_id": doc.get("id", f"chunk_{i}"),
                "source": doc.get("metadata", {}).get(
                    "source",
                    doc.get("metadata", {}).get("source_file", "unknown"),
                ),
                "page": doc.get("metadata", {}).get("page", "unknown"),
                "score": doc.get("similarity_score"),
                "preview": doc["content"][:120] + "...",
            }
            for i, doc in enumerate(reterived_doc, start=1)
        ]
        return context, sources
    
    # Build Input (sys msg, history , question/query)
    def _build_messages(self, context: str, query: str) ->Tuple[Any, Any]:
        sys_prompt = f"""
        prompt...
        {context}
        """
        return SystemMessage(content=sys_prompt), HumanMessage(content=query)
    
    # Extract the cittaions from the answer 
    def _extract_citations(self, answer: str) -> List[int]:
        matches = re.findall(r"\[(\d+)\]|【(\d+)】", answer)
        return sorted({int(a or b) for a, b in matches})
    
    def _resolve_sources(
        self, answer: str, sources: List[Dict]
    ) -> List[Dict]:
        citation_map = {source["citation_id"]: source for source in sources}
        used_citations = self._extract_citations(answer)

        return [
            citation_map[citation_id]
            for citation_id in used_citations
            if citation_id in citation_map
        ]
        
    def query_fun(self, query: str, top_k: int = 5, threshold: float = 0.0, stream: bool = False, summarize: bool = False)-> tuple[Dict, Dict, Dict]:
        # 1. reteriver
        reterived_doc = self._reterive_doc(
            query=query, top_k=top_k, threshold=threshold
        )

        if not reterived_doc:
            print("\n[INFO] No Context Found")
            answer = "I don't have enough information in the provided documents."

            self.history.append(HumanMessage(content=query))
            self.history.append(AIMessage(content=answer))

            return {
                "question": query,
                "answer": answer,
                "sources": [],
            }

        
        # 2. build context and sources from reterived_doc
        context, sources = self._build_context(reterived_doc)
        
        # 3. set a fallback technique in case for no reterived doc 
        if not context:
            print("No Context found")
            answer = ""
            sources = []
            
        
        # 4. Buid the Message(System Message and Human Message)
        sys_message, human_message = self._build_messages(context, query)
        
        # 5. Build the execution payload (combine history + SystemMessage + query)
        input_message = [sys_message] + self.history + [human_message]
        
        # 6. Model Generation 
        if stream:
            for chunk in self.agent.stream({"messages": input_message}):
                if chunk:
                    answer += chunk.content
                    print(chunk.content, end = "", flush=True)
        else:
            resp = self.agent.invoke({"messages": input_message})
            if isinstance(resp, dict) and "messages" in resp:
                answer = resp["messages"][-1].content
            if hasattr(resp, "content"):
                answer = resp.content
            else:
                answer = str(resp)
        
        # 7. resolve used sources from the citations 
        used_sources = self._resolve_sources(answer, sources)
        
        # 8. update the history 
        self.history.append(sys_message)
        self.history.append(human_message)
        
        # 7. return eturn {'question', 'answer', 'sources', 'history'}
        return {
            "question": query,
            "answer": answer,
            "source": used_sources,
            "history": self.history
        }


# llm

llm = ChatGroq(model="openai/gpt-oss-120b",temperature=0.0, max_tokens=1024, timeout=60, max_retries=2)
llm
summarizer = SummarizationMiddleware(
    model=llm,
    trigger=("tokens", 10000),
    keep=("messages", 4)
)
agent = create_agent(
    model=llm,
    tools=[],
    middleware=[summarizer]
)     
rag = AdvanceRagImplementation(agent=agent, reteriver=reterival)    


In [19]:
res = rag.query_fun("what are the characteristics of Vivekanad")

Batches: 100%|██████████| 1/1 [00:00<00:00, 67.31it/s]


Reterived 5


In [21]:
res["answer"]

'{\'messages\': [SystemMessage(content=\'\\n        prompt...\\n        [1]\\nVIVEKANAND S NAIK  \\nSoftware Engineer | Full Stack & Backend Developer | AI/ML Systems \\n+91 8217516956   |   Vivekanandnaik333@gmail.com   |   github.com/Vivekanand-s-naik   |   linkedin.com/in/Vivekanand-naik \\n \\nSUMMARY  \\nComputer Science undergraduate skilled in building full-stack and AI-powered backend systems with Python, Java, and modern web \\nframeworks. Experienced designing REST APIs, integrating computer vision/ML models into production-style pipelines, and architecting scalable \\nserver-side systems. Strong foundation in DSA, databases, and system design. \\nTECHNICAL SKILLS  \\nLanguages: Java (Core & Advanced), Python, JavaScript, SQL, C \\nBackend & Web: Spring Boot, Flask, FastAPI, REST API Design, HTML5, CSS3 \\nAI / Machine Learning & Computer Vision: TensorFlow, Keras, PyTorch, OpenCV, MediaPipe, dlib, face_recognition, YOLO, LangChain \\nDatabases: MySQL, PostgreSQL, MongoDB \\n